# SportsMOT: all trackers, default params, both state estimators (submission files)

In [1]:
import os
import shutil
import inspect
from collections import defaultdict

import numpy as np
import pandas as pd
import supervision as sv

from trackers import SORTTracker, OCSORTTracker, ByteTrackTracker
from trackers.utils.state_representations import XYXYStateEstimator, XCYCSRStateEstimator

SPORTSMOT_DET_ROOT = "sportsmot_yolox_dets"

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame.get(frame_id, []):
        det = line.split(",")
        x1, y1, x2, y2, conf = float(det[1]), float(det[2]), float(det[3]), float(det[4]), float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def make_tracker(tracker_name, state_estimator_class):
    cls_map = {
        "SORT": SORTTracker,
        "OCSORT": OCSORTTracker,
        "ByteTrack": ByteTrackTracker,
    }
    tracker_cls = cls_map[tracker_name]
    kwargs = {}
    try:
        params = inspect.signature(tracker_cls.__init__).parameters
        if "state_estimator_class" in params:
            kwargs["state_estimator_class"] = state_estimator_class
    except Exception:
        pass
    return tracker_cls(**kwargs)


def run_tracker_sportsmot_submission(tracker_name, estimator_name, estimator_class):
    split = "test"
    tracker = make_tracker(tracker_name, estimator_class)
    det_root = os.path.join(SPORTSMOT_DET_ROOT, split)

    outputs_root = f"{tracker_name}_outputs_sportsmot_default_estimators"
    save_dir = os.path.join(outputs_root, f"{split}_{estimator_name.lower()}")
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue
        tracker.reset()
        seq_name = seq.split(".")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []
        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue
                width = right - left
                height = bottom - top
                vertical = width / max(height, 1e-6) > VERTICAL_RATIO_THRESH
                if width * height > MIN_BOX_AREA and not vertical:
                    output_lines.append(
                        f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                    )

        with open(os.path.join(save_dir, seq_name + ".txt"), "w") as f:
            f.writelines(output_lines)

    zip_base = f"{tracker_name.lower()}_{os.path.basename(save_dir)}"
    zip_path = shutil.make_archive(zip_base, "zip", save_dir)
    return {"tracker": tracker_name, "state_estimator": estimator_name, "submission_dir": save_dir, "zip_path": zip_path}

In [2]:
estimators = [
    ("XYXY", XYXYStateEstimator),
    ("XCYCSR", XCYCSRStateEstimator),
]
trackers = ["SORT", "OCSORT", "ByteTrack"]

rows = []
for tracker_name in trackers:
    for estimator_name, estimator_class in estimators:
        print(f"Building SportsMOT submission for {tracker_name} with {estimator_name}...")
        rows.append(run_tracker_sportsmot_submission(tracker_name, estimator_name, estimator_class))

submission_df = pd.DataFrame(rows).sort_values(["tracker", "state_estimator"]).reset_index(drop=True)
submission_df

Building SportsMOT submission for SORT with XYXY...
Building SportsMOT submission for SORT with XCYCSR...
Building SportsMOT submission for OCSORT with XYXY...
Building SportsMOT submission for OCSORT with XCYCSR...
Building SportsMOT submission for ByteTrack with XYXY...
Building SportsMOT submission for ByteTrack with XCYCSR...


,tracker,state_estimator,submission_dir,zip_path
0,ByteTrack,XCYCSR,ByteTrack_outputs_sportsmot_default_estimators...,/Users/alexanderbodner/Documents/roboflow/trac...
1,ByteTrack,XYXY,ByteTrack_outputs_sportsmot_default_estimators...,/Users/alexanderbodner/Documents/roboflow/trac...
2,OCSORT,XCYCSR,OCSORT_outputs_sportsmot_default_estimators/te...,/Users/alexanderbodner/Documents/roboflow/trac...
3,OCSORT,XYXY,OCSORT_outputs_sportsmot_default_estimators/te...,/Users/alexanderbodner/Documents/roboflow/trac...
4,SORT,XCYCSR,SORT_outputs_sportsmot_default_estimators/test...,/Users/alexanderbodner/Documents/roboflow/trac...
5,SORT,XYXY,SORT_outputs_sportsmot_default_estimators/test...,/Users/alexanderbodner/Documents/roboflow/trac...
